# Beam pre-processing experiment

## Data loading

In [1]:
import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt('data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

selected_campaigns = list(range(1, 10))
# Data filtering
df = filter_dataframe(
    df=df,
    operators=[10],
    include_columns=['pci', 'beam_index', 'nr_arfcn', 'operator_id', 'rsrq'],
    campaigns=selected_campaigns,
)

Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [2]:
from typing import Optional, Tuple
from scripts.utils import RF_PARAM_5G
import pandas as pd


def get_best_pci_beam_pairs(
        mat: pd.DataFrame, rf_param: RF_PARAM_5G
) -> Tuple[Optional[Tuple[int, int]], Optional[int]]:
    # Drop rows where rf_param is NaN
    mat = mat.dropna(subset=[rf_param.value])

    # Check if the DataFrame is empty after dropping NaNs
    if mat.empty:
        print("No valid data available after dropping NaN values.")
        return None, None

    # Get the best beams by grouping only by 'pci' and 'operator_id'
    idx = mat.groupby(["pci"])[rf_param.value].idxmax()

    # Use the indices to select the rows with the highest 'rsrq' for each group
    return mat.loc[idx][['pci', 'beam_index']].values


df['beam_match'] = df['measurements_matrix'].apply(lambda x: get_best_pci_beam_pairs(x, RF_PARAM_5G.RSRQ))



In [3]:
df.iloc[0]['beam_match']

array([[-111,    3],
       [-109,    1],
       [-108,    5],
       [ -65,    6],
       [ -62,    4],
       [ -58,    3],
       [  10,    7],
       [  57,    3],
       [  75,    0],
       [  76,    7],
       [ 121,    1]], dtype=int8)